## 정확한 답변을 생성했는가? - 사내 문서 Q&A 에이전트 평가 (In-house Agent Evaluation)
* LangSmith를 사용하여 체계적인 평가를 수행
* 평가 프로세스
```
Golden Dataset (LangSmith)
        │
        ▼
┌─────────────────────────────┐
│   run_agent_to_completion   │  ← 에이전트 실행
└─────────────────────────────┘
        │
        ▼
┌─────────────────────────────┐
│      Evaluators 실행         │
│  • 답변 정확성 (LLM Judge)     │
└─────────────────────────────┘
        │
        ▼
    평가 결과 (LangSmith UI)
```

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langsmith import Client

# LangSmith 클라이언트 - 데이터셋 관리, 실험 실행, 결과 조회에 사용
ls_client = Client

In [3]:
# sampled golden dataset
import pandas as pd
dataset = pd.read_csv('cumulative_golden_dataset.csv')
dataset.head()

,question,answer,source
0,전결 규정 (Delegation of Authority)의 목적은 무엇인가요?,회사 내 의사 결정의 신속성과 책임성을 확보하기 위하여 전결 권한을 명확히 정하는 ...,delegation_of_authority
1,"인사(채용, 승진, 징계) 사항에 대한 최종 승인권자는 누구입니까?",인사 사항에 대한 최종 승인권자는 대표이사입니다.,delegation_of_authority
2,50만원 이하의 예산 승인은 누가 할 수 있나요?,"팀장과 부서장이 승인할 수 있으며, 본부장에게는 보고가 이루어집니다.",delegation_of_authority
3,200만원을 초과하는 예산 승인의 결재 라인은 어떻게 되나요?,"팀장 검토, 부서장 검토, 본부장 승인을 거쳐 대표이사가 최종 승인합니다.",delegation_of_authority
4,500만원을 초과하는 계약 체결 시 본부장의 역할은 무엇인가요?,"본부장은 해당 계약 건에 대해 승인 역할을 수행하며, 이후 대표이사의 최종 승인이 ...",delegation_of_authority


In [4]:
dataset['source'].value_counts()

source
employee_benefits_and_welfare_faq      13
delegation_of_authority                10
employee_benefits_and_welfare_guide    10
employee_handbook_and_hr_policy        10
expense_management_guide               10
it_support_guide                       10
legal_and_compliance_policy            10
Name: count, dtype: int64

In [5]:
dataset_sp = dataset.groupby(['source']).sample(3)
dataset_sp = dataset_sp.reset_index(drop=True)

In [6]:
dataset_sp.to_csv('./sampled_golden_dataset.csv',index=False)

## Grader (채점기) 설정
* LLM-as-Judge 방식으로 에이전트의 답변을 평가
* LLM이 "선생님" 역할을 하여 학생(에이전트)의 답변을 정답(ground truth)과 비교하고, 이진 점수(1=정답, 0=오답)를 부여

* 채점 기준:
    - 정답 대비 사실적 정확성만 평가
    - 모순되는 진술이 없어야 함
    - 정답보다 더 많은 정보를 포함해도, 사실적으로 정확하다면 OK

In [7]:
grader_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) RESPONSE, and the STUDENT RESPONSE.

Here is the grade criteria to follow:
(1) Grade the student responses based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student response does not contain any conflicting statements.
(3) It is OK if the student response contains more information than the ground truth response, 
    as long as it is factually accurate relative to the ground truth response.

Correctness:
1 means that the student's response meets all of the criteria.
0 means that the student's response does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct."""

In [8]:
from pydantic import BaseModel, Field

class Grade(BaseModel):
    """LLM Judge가 반환하는 채점 결과 스키마"""
    correctness: int = Field(description="1 if the student's response aligns with the content of the ground truth, 0 otherwise.")
    reasoning: str = Field(description="A step-by-step explanation of the grading decision.")

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-pro-preview",
        temperature=1
    )
grader_llm = llm.with_structured_output(Grade)

## Agent 실행함수 정의
* run_agent_to_completion >> LangSmith의 evaluate()가 각 테스트 케이스마다 호출하는 함수. Golden Dataset의 질문을 받아 에이전트를 실행하고, 결과를 반환함

In [10]:
from langchain_core.messages import HumanMessage
from inhouse_agent import agent  # 사내 문서 Q&A 에이전트 (agents/ 디렉토리에 정의)

def run_agent_to_completion(inputs):
    """LangSmith evaluate()가 호출하는 에이전트 실행 함수.
    
    Args:
        inputs: Golden Dataset의 한 행 (예: {"question": "연차 휴가는 며칠?"})
    
    Returns:
        에이전트 실행 결과 (messages 리스트 포함)
    """
    question = inputs["question"]
    
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    return result

### Evaluator 함수 정의
* Evaluator : 에이전트의 출력을 정답과 비교하여 점수를 매기는 함수
* LangSmith의 evaluate()에 전달되면, 각 테스트 케이스마다 자동으로 호출
* evaluator는 이진 평가 (1 또는 0)를 수행
    * inputs: 질문
    * outputs: 에이전트의 응답 (messages[-1]이 최종 답변)
    * reference_outputs: Golden Dataset의 정답

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage

def is_final_answer_correct(inputs, outputs, reference_outputs):
    """LLM-as-Judge로 에이전트 답변의 정확성을 이진 평가하는 evaluator.
    
    Args:
        inputs: 원본 질문 (Golden Dataset의 inputs)
        outputs: 에이전트 실행 결과 (run_agent_to_completion의 반환값)
        reference_outputs: 정답 (Golden Dataset의 outputs)
    
    Returns:
        int: 1(정답) 또는 0(오답)
    """
    question = inputs["question"]
    answer = outputs["messages"][-1].content  # 에이전트의 최종 답변 (마지막 메시지)
    ground_truth_response = reference_outputs["answer"]  # Golden Dataset의 정답
    
    # 질문, 에이전트 답변, 정답을 하나의 문자열로 포맷팅하여 LLM Judge에 전달
    grading_input = f"""QUESTION: {question}
ANSWER: {answer}
GROUND TRUTH: {ground_truth_response}"""
    
    # LLM Judge가 Grade 스키마(correctness + reasoning)로 채점
    grade = grader_llm.invoke([
        SystemMessage(content=grader_instructions),
        HumanMessage(content=grading_input)
    ])

    return grade.correctness  # 1 또는 0 반환

In [12]:
# 실험
# 1. Golden Dataset에서 각 예제를 순회
# 2. run_agent_to_completion으로 에이전트 실행
# 3. is_final_answer_correct evaluator로 결과 채점
# 4. LangSmith 대시보드에 결과 기록

In [13]:
from langsmith import Client

# LangSmith 클라이언트 - 데이터셋 관리, 실험 실행, 결과 조회에 사용
ls_client = Client()
dataset = ls_client.read_dataset(dataset_name="agent-eval-sampled")

In [14]:
dataset

Dataset(name='agent-eval-sampled', description='sampled golden dataset', data_type=<DataType.kv: 'kv'>, id=UUID('e6169a73-73fd-4f8e-ab1b-cb8e5dad6485'), created_at=datetime.datetime(2026, 6, 15, 15, 41, 36, 619953, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 15, 15, 41, 36, 619953, tzinfo=TzInfo(0)), example_count=21, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata=None)

In [15]:
experiment_result = ls_client.evaluate(
    run_agent_to_completion,            # 평가 대상: 에이전트 실행 함수
    data="agent-eval-sampled",          # Golden Dataset 이름
    evaluators=[is_final_answer_correct],  # 평가자: 이진 정확성 evaluator
    max_concurrency=2,                  # 동시 실행 수 (API rate limit 고려)
    num_repetitions=1                   # 각 예제당 1회 실행
)

/Users/a202304035/LLM_Eval_study/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'terrific-kiss-14' at:
https://smith.langchain.com/o/2f003b67-27f4-48d0-8856-e1db2397e3d4/datasets/e6169a73-73fd-4f8e-ab1b-cb8e5dad6485/compare?selectedSessions=576ee43d-0894-485c-8147-6aa41eec599d




21it [03:41, 10.57s/it]
